# 以機器學習預測口腔癌篩檢異常結果

**課程**：115b 機器學習（NDHU）｜**學期**：2025–2026 下學期

## 使用前準備

1. 將 `oral0507-v1.csv` 上傳至 Google Drive，建議路徑：`我的雲端硬碟/oral-cancer/oral0507-v1.csv`
2. 若放置路徑不同，修改 **Cell 5** 的 `DRIVE_CSV_PATH`
3. 執行 **Runtime → Run all**

---

In [ ]:
# Cell 1｜掛載 Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 2｜Clone GitHub repo
import os
if not os.path.exists('/content/oral-cancer'):
    !git clone https://github.com/petertseng0517/Oral-Cancer-Screening-Abnormality-Prediction.git /content/oral-cancer
else:
    !git -C /content/oral-cancer pull
    print('Repo already exists, pulled latest changes.')

In [ ]:
# Cell 3｜切換工作目錄 & 安裝套件
# 版本鎖定與 requirements.txt 一致，確保 Colab 與本機算出來的模型指標數字可完全重現
# （sklearn/numpy 等套件版本不同，即使 random_state 相同，數值仍可能有第3、4位小數的差異）
%cd /content/oral-cancer
!pip install -q pandas==3.0.3 numpy==2.4.6 scikit-learn==1.8.0 imbalanced-learn==0.14.1 matplotlib==3.10.9 seaborn==0.13.2 openpyxl==3.1.5

# 注意：若安裝完看到「RESTART RUNTIME」提示，請點擊重新啟動工作階段後，
# 從頭重新執行 Run all（讓新版套件確實載入到記憶體中）

In [ ]:
# Cell 4｜安裝中文字型（matplotlib 圖表顯示用）
# 先 apt-get update 確保套件索引最新，再安裝（不隱藏輸出，方便排查問題）
!apt-get update -qq
!apt-get install -y fonts-noto-cjk

# 驗證字型檔是否確實安裝成功
# 註：實際繪圖在 Cell 6 以「獨立 Python 行程」執行 main.py，
#    該行程會在 import matplotlib 時自動重新掃描系統字型並偵測到剛裝好的 Noto CJK，
#    因此這裡僅作驗證、不需要額外用 addfont() 註冊
import glob
font_files = glob.glob('/usr/share/fonts/**/*Noto*CJK*', recursive=True)
if font_files:
    print(f'✅ 已偵測到 {len(font_files)} 個 Noto CJK 字型檔，main.py 執行時將自動套用於圖表')
else:
    print('⚠️ 未偵測到字型檔，圖表中文可能無法正常顯示，請檢查上方安裝紀錄')

In [ ]:
# Cell 5｜從 Google Drive 複製資料檔
# ↓ 若資料檔放在不同路徑，修改這裡
DRIVE_CSV_PATH = '/content/drive/MyDrive/oral-cancer/oral0507-v1.csv'

import shutil
shutil.copy(DRIVE_CSV_PATH, '/content/oral-cancer/oral0507-v1.csv')
print(f'資料檔已複製：{DRIVE_CSV_PATH} → /content/oral-cancer/oral0507-v1.csv')

In [ ]:
# Cell 6｜執行訓練與評估（需 3–5 分鐘）
!python main.py

In [ ]:
# Cell 7｜顯示評估圖表
from IPython.display import Image, display
import os

output_dir = 'data/processed/'
figures = [
    ('fig_confusion_matrix.png',  'Confusion Matrix'),
    ('fig_roc_curve.png',         'ROC 曲線'),
    ('fig_feature_importance.png','特徵重要性'),
    ('fig_abnormal_position.png', '異常部位分布'),
]
for filename, title in figures:
    path = os.path.join(output_dir, filename)
    if os.path.exists(path):
        print(f'\n── {title}')
        display(Image(path))
    else:
        print(f'找不到 {filename}，請確認 Cell 6 是否正常完成')

In [ ]:
# Cell 8｜顯示模型比較表
import pandas as pd
df = pd.read_csv('data/processed/model_comparison.csv')
display(df)

In [ ]:
# Cell 9｜（選用）下載輸出檔至本機
from google.colab import files

for f in ['data/processed/model_bundle.pkl', 'data/processed/model_comparison.csv']:
    if os.path.exists(f):
        files.download(f)

In [ ]:
# Cell 10｜單筆推論 Demo：執行 demo_predict.py（兩組對照案例：低風險／正常 vs. 高風險／異常）
# 與本機 VS Code 展示的是同一支程式，確保兩邊 Demo 畫面一致
!python demo_predict.py